# 02 Compare Single Level

In [ ]:
# User-editable papermill/environment configuration
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/tmp/wwpgd_v2/data"))
RESULTS_ROOT = Path(
    os.environ.get(
        "WWGPT_RESULTS_ROOT",
        os.environ.get(
            "RESULTS_ROOT",
            "/tmp/wwpgd_v2/real_level0_results_v5/"
            "experiments/level_00/multiplier_20",
        ),
    )
)
RUN_LOG = Path(os.environ.get("RUN_LOG", "/tmp/wwpgd_v2/real_level0_five_seed_v4.log"))
PID_FILE = Path(os.environ.get("PID_FILE", "/tmp/wwpgd_v2/real_level0_five_seed_v4.pid"))

ANALYSIS_DIR = RESULTS_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_LEGACY = False
EXPECTED_SEEDS = {1337, 2027, 4099, 7919, 104729}


In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd().resolve()
repo = cwd if (cwd/'src'/'wwgpt').exists() else cwd.parent
if str(repo/'src') not in sys.path:
    sys.path.insert(0, str(repo/'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from wwgpt.analysis import *
RESULTS_ROOT = resolve_experiment_root(RESULTS_ROOT)
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Run log: {RUN_LOG}')
print(f'PID file: {PID_FILE}')


Canonical pair discovery, run inventory, curves, paired terminal results, compute, and projection diagnostics.

In [ ]:
candidates = discover_pair_candidates(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
canonical_pairs, pair_audit = select_canonical_pairs(candidates)
runs = discover_canonical_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
inventory = build_run_inventory(runs)
pair_audit.to_csv(ANALYSIS_DIR/'pair_candidate_audit.csv', index=False)
inventory.to_csv(ANALYSIS_DIR/'run_inventory.csv', index=False)
print(f'Canonical pairs: {len(canonical_pairs)}')
display(pair_audit)
display(inventory)


In [ ]:
def plot_metric(metric, filename, ylabel=None, final_frac=None):
    fig, ax = plt.subplots(figsize=(7,4))
    byfam={}
    for r in runs:
        m=load_run_artifacts(Path(r['run_dir']))['metrics']
        if metric not in m or 'tokens_seen' not in m: continue
        if final_frac is not None:
            cutoff=m['tokens_seen'].max()*(1-final_frac); m=m[m['tokens_seen']>=cutoff]
        ax.plot(m['tokens_seen'], m[metric], alpha=.25, color=OPTIMIZER_COLORS[r['optimizer_family']])
        byfam.setdefault(r['optimizer_family'],[]).append(m)
    for fam,curves in byfam.items():
        grid, vals=align_curves(curves,'tokens_seen',metric)
        if len(grid):
            mean=vals.mean(axis=0); sem=vals.std(axis=0,ddof=1)/np.sqrt(vals.shape[0]) if vals.shape[0]>1 else 0
            t=stats.t.ppf(.975, vals.shape[0]-1) if vals.shape[0]>1 else 0
            ax.plot(grid,mean,lw=3,label=OPTIMIZER_LABELS[fam],color=OPTIMIZER_COLORS[fam])
            if vals.shape[0]>1: ax.fill_between(grid,mean-t*sem,mean+t*sem,alpha=.15,color=OPTIMIZER_COLORS[fam])
    ax.set_xlabel('tokens_seen'); ax.set_ylabel(ylabel or metric); ax.legend(); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/filename); plt.close(fig)

fields=['step','tokens_seen','unique_tokens_seen','realized_tokens','tokens_per_parameter','train_loss','validation_loss','learning_rate','gradient_norm','parameter_norm','update_norm','elapsed_seconds','training_seconds','evaluation_seconds','estimated_flops','tokens_per_second','evaluation_token_count','evaluation_batches','validation_probe_hash','training_probe_hash','projection_event','scheduled_token_fraction','actual_step','actual_tokens_seen','projection_runtime','layer_name','hardness','relative_frobenius_change','xmin','detX_num','tail_size','TraceLog_before','TraceLog_after','skip_reason','initialization_hash','data_hash','tokenizer_hash','git_commit','hardware','PyTorch version','WeightWatcher version']
allcols=set(inventory.columns)
for r in runs:
    art=load_run_artifacts(Path(r['run_dir'])); allcols |= set(art['metrics'].columns)|set(art['projection'].columns)|set(art['spectral'].columns)|set(art['manifest'].keys())
coverage=pd.DataFrame([{'field':f,'status':'available' if f in allcols else 'missing'} for f in fields]); coverage.to_csv(ANALYSIS_DIR/'logging_coverage.csv',index=False); display(coverage)
plot_metric('validation_loss','validation_loss_vs_tokens.png','validation loss')
plot_metric('train_loss','training_loss_vs_tokens.png','training loss')
plot_metric('validation_loss','validation_loss_final_30_percent.png','validation loss',final_frac=.30)
term=terminal_results(runs); term.to_csv(ANALYSIS_DIR/'paired_terminal_results.csv',index=False); display(term)
fig,ax=plt.subplots(); term[['adamw_final_validation_loss','wwpgd_final_validation_loss']].plot(kind='bar',ax=ax); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'paired_final_validation_loss.png'); plt.close(fig)
fig,ax=plt.subplots(); term['wwpgd_minus_adamw_validation_loss'].plot(kind='bar',ax=ax); ax.axhline(0,color='k'); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'paired_final_loss_difference.png'); plt.close(fig)
pairs=[]
for seed,g in pd.DataFrame(runs).groupby('seed'):
    a=g[g.optimizer_family=='adamw'].iloc[0]; w=g[g.optimizer_family=='wwpgd'].iloc[0]
    pairs.append((load_run_artifacts(Path(a.run_dir))['metrics'],load_run_artifacts(Path(w.run_dir))['metrics']))
grid,diffs=paired_curve_differences(pairs,'tokens_seen','validation_loss')
fig,ax=plt.subplots();
if len(grid):
    ax.plot(grid,diffs.T,alpha=.25,color='gray'); ax.plot(grid,diffs.mean(axis=0),lw=3,color='black'); ax.axhline(0,color='k',ls='--')
ax.set_title('Negative favors AdamW + WW-PGD'); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'paired_validation_loss_difference_vs_tokens.png'); plt.close(fig)
projs=[]
for r in runs:
    p=load_run_artifacts(Path(r['run_dir']))['projection']
    if not p.empty: p=p.assign(seed=r['seed'],pair_id=r['pair_id']); projs.append(p)
proj=pd.concat(projs,ignore_index=True) if projs else pd.DataFrame(); proj.to_csv(ANALYSIS_DIR/'wwpgd_projection_records.csv',index=False)
if not proj.empty:
    ev=proj.groupby(['seed','projection_event']).agg(rows=('layer_name','size'),changed_layers=('changed','sum'),tokens_seen=('tokens_seen','first'),projection_runtime=('projection_runtime','sum')).reset_index(); ev.to_csv(ANALYSIS_DIR/'wwpgd_projection_event_summary.csv',index=False)
    lay=proj.groupby('layer_name').agg(rows=('layer_name','size'),mean_change=('relative_frobenius_change','mean')).reset_index(); lay.to_csv(ANALYSIS_DIR/'wwpgd_projection_layer_summary.csv',index=False)
    for x,y,f in [('projection_event','tokens_seen','projection_schedule.png'),('projection_event','relative_frobenius_change','relative_projection_norm_by_event.png'),('projection_event','projection_runtime','projection_runtime_by_event.png'),('projection_event','trace_log_delta','tracelog_preservation.png')]:
        fig,ax=plt.subplots(); ax.scatter(proj[x],proj[y]); ax.set_xlabel(x); ax.set_ylabel(y); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/f); plt.close(fig)
print('Exact per-projection alpha_before and alpha_after were not logged. Alpha trajectories are analyzed from spectral.csv in notebook 03.')
